# Catalog Setup

Creates the Unity Catalog structure for the
Databricks GenAI Data Analyst Copilot.

Architecture:

genai_copilot
├── bronze
├── silver
└── gold

In [0]:
# Check the current catalog and user context.

print("Current catalog:")
display(spark.sql("SELECT current_catalog()"))

print("Current schema:")
display(spark.sql("SELECT current_schema()"))

In [0]:
# Create the project catalog if the workspace permits it.

spark.sql("""
CREATE CATALOG IF NOT EXISTS genai_copilot
""")

print("Catalog creation command completed.")

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS genai_copilot.bronze
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS genai_copilot.silver
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS genai_copilot.gold
""")

print("Bronze, Silver and Gold schemas are ready.")

In [0]:
display(
    spark.sql("""
        SHOW SCHEMAS IN genai_copilot
    """)
)

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS genai_copilot.bronze.bronze_files
""")

print("Bronze volume created.")

In [0]:
volume_path = "/Volumes/genai_copilot/bronze/bronze_files"

print(volume_path)

In [0]:
import os

sales_file = "/Volumes/genai_copilot/bronze/bronze_files/sales.csv"

print("File exists:", os.path.exists(sales_file))
print("File:", sales_file)

In [0]:
raw_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(sales_file)
)

print("Rows:", raw_df.count())
print("Columns:", len(raw_df.columns))

display(raw_df.limit(10))

In [0]:
raw_df.printSchema()

In [0]:
from pyspark.sql import functions as F

bronze_df = (
    raw_df
    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "source_file",
        F.lit("sales.csv")
    )
)

display(bronze_df.limit(10))

In [0]:
bronze_table = "genai_copilot.bronze.sales_raw"

(
    bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(bronze_table)
)

print(f"Created Bronze table: {bronze_table}")

In [0]:
%sql
SELECT *
FROM genai_copilot.bronze.sales_raw
LIMIT 10;

In [0]:
%sql
SELECT COUNT(*) AS row_count
FROM genai_copilot.bronze.sales_raw;

In [0]:
%sql
SELECT
    order_id,
    COUNT(*) AS occurrences
FROM genai_copilot.bronze.sales_raw
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY occurrences DESC;

In [0]:
%sql
SELECT
    COUNT(*) AS invalid_quantity_rows
FROM genai_copilot.bronze.sales_raw
WHERE quantity <= 0;

In [0]:
%sql
SELECT
    COUNT(*) AS invalid_discount_rows
FROM genai_copilot.bronze.sales_raw
WHERE discount < 0
   OR discount > 1;

In [0]:
%sql
SELECT
    COUNT(*) AS missing_customer_names
FROM genai_copilot.bronze.sales_raw
WHERE customer_name IS NULL;

In [0]:
%sql
SELECT
    source_file,
    MIN(ingestion_timestamp) AS first_ingestion,
    MAX(ingestion_timestamp) AS last_ingestion,
    COUNT(*) AS rows
FROM genai_copilot.bronze.sales_raw
GROUP BY source_file;

In [0]:
%sql
SELECT COUNT(*)
FROM genai_copilot.bronze.sales_raw;

In [0]:
%sql
SELECT *
FROM genai_copilot.bronze.sales_raw
LIMIT 10;